# Demo 4：Flow Reaction Optimization

任务：`flow-reaction-optimization`。本 Demo 将流量、停留时间和温度视为动力学系统辨识干预，观察隐藏速率律与传递边界怎样改变转化率和风险。

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'chemworld').exists()), None)
if ROOT is None:
    raise RuntimeError('请从 ChemWorld 仓库内启动 notebook')
for path in (ROOT, ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
du = importlib.import_module('notebooks.task_demos.demo_utils')

TASK_ID = 'flow-reaction-optimization'
SEED = 0

## 1. 公开任务合同

Agent 能设置公开流量、停留时间与运行温度，并读取 UV-vis/final assay 反馈；不读取 hidden state、真实速率常数或内部温度场。

In [2]:
card = du.task_card(TASK_ID)
pd.DataFrame({
    'field': ['task_id', 'motivation', 'budget', 'episode_mode', 'success_metrics'],
    'value': [card['task_id'], card['scientific_motivation'], card['budget'], card['episode_mode'], card['reward_leaderboard_metric']['success_metrics']],
})

,field,value
0,task_id,flow-reaction-optimization
1,motivation,Optimize a geometry-resolved PFR using the sha...
2,budget,60
3,episode_mode,campaign
4,success_metrics,"[score, flow_conversion, yield, safety_risk]"


## 2. 构造候选干预

三组候选条件覆盖不同流量、停留时间和温度区域。完整 recipe 由公开 adapter 生成。

In [3]:
vectors = du.standard_probe_vectors(TASK_ID)
mid_recipe = du.recipe_frame(TASK_ID, vectors['mid'])
mid_recipe

,step,operation,payload
0,1,add_solvent,"{'operation': 'add_solvent', 'volume_L': 0.025..."
1,2,add_reagent,"{'operation': 'add_reagent', 'amount_mol': 0.0..."
2,3,add_catalyst,"{'operation': 'add_catalyst', 'catalyst_amount..."
3,4,set_flow_rate,"{'operation': 'set_flow_rate', 'flow_rate_mL_m..."
4,5,run_flow,"{'operation': 'run_flow', 'target_temperature_..."
5,6,measure,"{'operation': 'measure', 'instrument': 'uvvis'}"
6,7,terminate,{'operation': 'terminate'}
7,8,measure,"{'operation': 'measure', 'instrument': 'final_..."


## 3. 执行并读取反馈

比较 `flow_conversion`、`yield`、`safety_risk`、资源成本和 `leaderboard_score`。这只是环境响应面，不是训练过程。

In [4]:
comparison = du.compare_vectors(TASK_ID, vectors, seed=SEED)
columns = ['candidate', 'leaderboard_score', 'flow_conversion', 'yield', 'safety_risk', 'cost', 'all_actions_valid']
comparison[[column for column in columns if column in comparison]]

,candidate,leaderboard_score,flow_conversion,yield,safety_risk,cost,all_actions_valid
0,low,0.004300,0.000000,0.000000,None,0.0,True
1,mid,0.030213,0.013042,0.015055,None,0.0,True
2,high,0.030543,0.058641,0.052832,None,0.0,True


### 查看流动运行后的测量证据

Agent 可以将公开 `processed_estimate` 与执行前预测比较。

In [5]:
run = du.run_vector(TASK_ID, vectors['mid'], seed=SEED)
measurement_feedback = du.measurement_trace(run)
measurement_feedback

,step,operation,reward,leaderboard_score,observed_keys,processed_estimate
0,6,measure,0.014192,NaN,"yield, conversion, flow_conversion, cost, safe...","{'yield': 0.028580852249596785, 'conversion': ..."
1,8,measure,0.016021,0.030213,"yield, selectivity, conversion, byproduct_sign...","{'yield': 0.015054836312189186, 'selectivity':..."


## 4. 同一干预，不同隐藏规律

World B 使用 `rate_law_family` 教学控制。公开设定完全相同，Agent 需要从转化与风险反馈判断当前动力学 world model 是否失效。

In [6]:
paired_worlds = du.compare_hidden_worlds(
    TASK_ID, vectors['mid'], mechanism_mode='rate_law_family', seed=SEED
)
paired_worlds

,opaque_world,score,flow_conversion,yield,safety_risk,leaderboard_score,cost,all_actions_valid
0,World A,None,0.013042,0.015055,None,0.030213,0.0,True
1,World B,None,0.000000,0.000000,None,0.000000,0.0,True


## 5. 留给 Agent 的能力问题

环境提供反馈；由外部 Agent 决定怎样将历史压缩到权重、上下文、belief 或在线模型中。

In [7]:
capability_probe = {
    'current_evidence': comparison.to_dict(orient='records'),
    'prediction_query': '预测一个未执行 flow-rate/residence-time/temperature 条件的转化与风险。',
    'next_intervention_query': '选择最能区分速率律变化与停留时间不足的下一次运行。',
    'evaluation_note': '衡量预测校准、辨识效率和反馈后的决策变化。',
}
capability_probe

{'current_evidence': [{'candidate': 'low',
   'score': None,
   'flow_conversion': 0.0,
   'yield': 0.0,
   'safety_risk': None,
   'leaderboard_score': 0.00429952119228919,
   'cost': 0.0,
   'all_actions_valid': True,
   'operation_count': 8},
  {'candidate': 'mid',
   'score': None,
   'flow_conversion': 0.013041632499776768,
   'yield': 0.015054836312189186,
   'safety_risk': None,
   'leaderboard_score': 0.030213040164293065,
   'cost': 0.0,
   'all_actions_valid': True,
   'operation_count': 8},
  {'candidate': 'high',
   'score': None,
   'flow_conversion': 0.058641005057875284,
   'yield': 0.05283207732936499,
   'safety_risk': None,
   'leaderboard_score': 0.03054318140671621,
   'cost': 0.0,
   'all_actions_valid': True,
   'operation_count': 8}],
 'prediction_query': '预测一个未执行 flow-rate/residence-time/temperature 条件的转化与风险。',
 'next_intervention_query': '选择最能区分速率律变化与停留时间不足的下一次运行。',
 'evaluation_note': '衡量预测校准、辨识效率和反馈后的决策变化。'}